# Step 1: Automatic cell segmentation with Cellpose

In [ ]:
import sys
sys.path.append("..")
import os
import tifffile as tiff
from matplotlib import pyplot as plt
from cellpose import models, io
import numpy as np

from utility import config
from utility import util
from utility.data_loader import DataLoader

## User input here
sub_ids = np.array([1,2,3,4,5])

def segment_all_frames(img):
    # Segmentation
    imgs = [img[i,:,:] for i in range(img.shape[0])]
    # Run cellpose
    model = models.CellposeModel(gpu=True)
    masks, flows, styles = model.eval(imgs)
    # Save masks as tif
    masks_all = np.stack(masks, axis=0)
    return masks_all


metadata = util.load_metadata()
metadata_sel = metadata[metadata['sub'].isin(sub_ids)].reset_index(drop=True)

for i in range(metadata_sel.shape[0]):
    sub_id = metadata_sel.iloc[i]['sub']
    print(f'Processing sub {sub_id}...')
    dl = DataLoader(metadata_sel.iloc[i])
    image = dl.load_data('xyProjection', 'tif')
    membrane_mask_path = dl.generate_file_path('memMask', 'tif')
    nucleus_mask_path = dl.generate_file_path('nucMask', 'tif')

    img_ch0 = image[:,0,:,:]
    img_ch1 = image[:,1,:,:]

    masks_all = segment_all_frames(img_ch1)
    tiff.imwrite(nucleus_mask_path, masks_all.astype(np.uint16))
 
    masks_all = segment_all_frames(img_ch0)
    tiff.imwrite(membrane_mask_path, masks_all.astype(np.uint16))